# 15. Exposure, replication, and variance decomposition

![Exposure and replication](../images/15_exposure_and_replication.svg)

This notebook takes one score tensor and shows how many different sample sizes live inside it. Fixed exposure sits on one axis, catalog support on another, participants on a third, and paired trained-model blocks on the axis that sets the primary standard error.

**Learning goals:** distinguish exposure from available support, keep participants separate from trained-model blocks, preserve complete allocation blocks during resampling, calculate the residual GFC-minus-completion contrast, and run an outcome-blind prospective simulation with eight blocks.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

SEED = 15
rng = np.random.Generator(np.random.PCG64(SEED))
np.set_printoptions(precision=3, suppress=True)
print(f"NumPy {np.__version__}; seed={SEED}")


## 1. Fixed exposure and changing support

Two counts are easy to confuse. **Exposure** counts the optimization draws a model receives. **Support** counts the distinct sequence-origin atoms those draws can reach. The experiment fixes the first and deliberately changes the second.

The active allocations all expose the model to the same number of sampled clips and the same nominal catalog size of 250,000 atoms. What changes is where the atoms live: breadth uses more sequences with one origin each, balanced uses fewer sequences with two origins each, and phase depth uses still fewer sequences with four origins each.

In [ ]:
n_blocks = 8
training_exposure = 8_192_000
allocations = np.array(["breadth", "balanced", "phase_depth"])
unique_sequences = np.array([250_000, 125_000, 62_500])
origins_per_sequence = np.array([1, 2, 4])
nominal_catalog = unique_sequences * origins_per_sequence
exposure_by_cell = np.full((n_blocks, 3), training_exposure, dtype=np.int64)
support_by_cell = np.broadcast_to(nominal_catalog, (n_blocks, 3))
recurrence = training_exposure / nominal_catalog
assert np.unique(exposure_by_cell).tolist() == [training_exposure]
assert np.unique(nominal_catalog).tolist() == [250_000]
assert np.allclose(recurrence, recurrence[0])
print(dict(zip(allocations, zip(unique_sequences, origins_per_sequence))))
print(f"recurrence per nominal atom={recurrence[0]:.2f}")


## 2. Participants and model blocks are different axes

Support and exposure describe training. The next question is how the resulting scores are organized, because the layout of the array decides which axis carries independence.

The primary outcome tensor has shape `(block, allocation, participant)`. The last axis holds repeated measurements on the same people, so it reduces measurement noise. The first axis holds eight matched training blocks, so it carries the model-level replication. Primary inference collapses the participant axis and leaves one contrast per block.

In [ ]:
B, P = 8, 308
gfc_expectation = np.array([0.30, 0.34, 0.41])
completion_expectation = np.array([0.23, 0.24, 0.25])
participant = rng.normal(0, 0.05, size=(1, 1, P))
block = rng.normal(0, 0.015, size=(B, 1, 1))
cell_noise_g = rng.normal(0, 0.010, size=(B, 3, 1))
cell_noise_c = rng.normal(0, 0.010, size=(B, 3, 1))
scores_g = np.clip(gfc_expectation[None, :, None] + participant + block + cell_noise_g + rng.normal(0, 0.025, (B, 3, P)), -1, 1)
scores_c = np.clip(completion_expectation[None, :, None] + 0.7 * participant + 0.6 * block + cell_noise_c + rng.normal(0, 0.025, (B, 3, P)), -1, 1)
assert scores_g.shape == scores_c.shape == (8, 3, 308)
print("GFC and completion shapes:", scores_g.shape, scores_c.shape)


## 3. Primary residual contrasts

With the tensor laid out, the primary estimate is a residual contrast inside each block. The function below averages participants within every allocation, subtracts independent completion from GFC on the same scale, and returns phase depth minus breadth.

This is not a pair of separate significance tests. The two scores are placed on one scale first, then differenced inside the block that produced them.

In [ ]:
def primary_contrasts(gfc_scores, completion_scores):
    gfc_means = np.asarray(gfc_scores, dtype=np.float64).mean(axis=-1)
    completion_means = np.asarray(completion_scores, dtype=np.float64).mean(axis=-1)
    residual = gfc_means - completion_means
    return residual[:, 2] - residual[:, 0]

def t_interval(values, confidence):
    values = np.asarray(values, dtype=np.float64)
    mean = values.mean()
    se = values.std(ddof=1) / np.sqrt(len(values))
    critical = stats.t.ppf((1 + confidence) / 2, df=len(values) - 1)
    return mean, (mean - critical * se, mean + critical * se)

P_r = primary_contrasts(scores_g, scores_c)
p_mean, p_interval_95 = t_interval(P_r, 0.95)
_, p_interval_90 = t_interval(P_r, 0.90)
margin = 0.0625
p_resolved = ((p_interval_95[0] > 0 or p_interval_95[1] < 0) and abs(p_mean) >= margin)
p_equivalent = p_interval_90[0] > -margin and p_interval_90[1] < margin
assert P_r.shape == (8,)
assert not (p_resolved and p_equivalent)
print(f"mean P={p_mean:.3f}; resolved={p_resolved}; equivalent={p_equivalent}")


## 4. Participant-only and crossed sensitivity bootstraps

The primary interval comes from eight numbers and a Student $t$ distribution. The two bootstraps below are sensitivities around that interval, and each one names a different population axis.

Participant-only resampling treats the eight trained-model blocks as fixed and carries every selected participant's full `(8, 3)` allocation profile. Crossed resampling also draws complete blocks. The block draw and the participant draw are applied as separate indexing operations so the resample keeps a Cartesian crossing rather than pairing unrelated index arrays.

In [ ]:
def bootstrap_primary(gfc_scores, completion_scores, replicates, rng, *, resample_blocks):
    gfc_scores = np.asarray(gfc_scores, dtype=np.float64)
    completion_scores = np.asarray(completion_scores, dtype=np.float64)
    B, A, P = gfc_scores.shape
    estimates = np.empty(replicates)
    for b in range(replicates):
        participant_draw = rng.integers(0, P, size=P, endpoint=False)
        sampled_g = gfc_scores[..., participant_draw]
        sampled_c = completion_scores[..., participant_draw]
        if resample_blocks:
            block_draw = rng.integers(0, B, size=B, endpoint=False)
            sampled_g = sampled_g[block_draw]
            sampled_c = sampled_c[block_draw]
        assert sampled_g.shape == sampled_c.shape == (B, A, P)
        estimates[b] = primary_contrasts(sampled_g, sampled_c).mean()
    return estimates

bootstrap_rng = np.random.Generator(np.random.PCG64(1501))
participant_only = bootstrap_primary(scores_g, scores_c, 2000, bootstrap_rng, resample_blocks=False)
crossed = bootstrap_primary(scores_g, scores_c, 2000, bootstrap_rng, resample_blocks=True)
participant_ci = np.quantile(participant_only, [0.025, 0.975], method="linear")
crossed_ci = np.quantile(crossed, [0.025, 0.975], method="linear")
assert participant_ci[0] < participant_ci[1] and crossed_ci[0] < crossed_ci[1]
print("participant-only interval:", participant_ci)
print("crossed interval:         ", crossed_ci)


## 5. Jitter is a separate diagnostic axis

Nearby jitter is paired with phase depth in four prespecified blocks. It is not part of the primary three-allocation path. That means it gets its own smaller tensor and its own lower-precision contrast.

The diagnostic asks whether separated phase origins beat local start variation after the same GFC-minus-completion residual is formed.

In [ ]:
J_BLOCKS = 4
phase_residual = (scores_g[:J_BLOCKS, 2].mean(axis=-1) - scores_c[:J_BLOCKS, 2].mean(axis=-1))
nearby_jitter_residual = phase_residual - rng.normal(0.045, 0.018, size=J_BLOCKS)
jitter_contrast = phase_residual - nearby_jitter_residual
jitter_mean, jitter_interval = t_interval(jitter_contrast, 0.95)
assert jitter_contrast.shape == (4,)
print(f"mean J={jitter_mean:.3f}; 95% interval={jitter_interval}")


## 6. The blocks share a finite corpus

One limit survives both bootstraps above, and it belongs beside every interval they produce.

The eight pool orderings overlap because they are drawn from one finite corpus. Resampling blocks therefore describes reproducibility over the declared pool ordering, phase construction, and optimization randomness, conditional on that corpus. It does not simulate eight independently sampled source datasets, and no number of bootstrap draws creates a new trained model or a new participant.

## 7. Prospective simulation keeps $n=8$

Everything so far analyzes data that exists. This last step happens before any of it exists, and it answers one question: can eight blocks resolve an effect worth reporting?

Simulate exactly eight primary contrast values under plausible design assumptions, apply the planned interval with seven degrees of freedom, and apply the exact materiality rule. The simulation reports how wide the interval tends to be, how often it excludes zero, and how often the estimate also reaches the 0.0625 margin. Outcome aggregates may not tune the assumed mean, assumed standard deviation, margin, or decision threshold.

In [ ]:
simulation_rng = np.random.Generator(np.random.PCG64(1502))
n_studies, n_model_blocks = 5000, 8
assumed_mean, assumed_sd = 0.08, 0.06
simulated = simulation_rng.normal(assumed_mean, assumed_sd, size=(n_studies, n_model_blocks))
means = simulated.mean(axis=1)
ses = simulated.std(axis=1, ddof=1) / np.sqrt(n_model_blocks)
critical = stats.t.ppf(0.975, df=n_model_blocks - 1)
lower, upper = means - critical * ses, means + critical * ses
resolved_positive = lower > 0
materially_positive = resolved_positive & (means >= margin)
assert simulated.shape == (5000, 8)
print(f"median 95% half-width={np.median(critical * ses):.3f}")
print(f"P(resolve above zero)={resolved_positive.mean():.3f}")
print(f"P(materially positive)={materially_positive.mean():.3f}")

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(means, bins=35, color="#6d9f71", edgecolor="white")
ax.axvline(0, color="black", label="zero")
ax.axvline(margin, color="#9f3f3f", label="materiality margin")
ax.set(xlabel="simulated mean primary contrast", ylabel="study count", title="Prospective eight-block simulation")
ax.legend()
plt.tight_layout()
plt.show()


## Exercises, limits, and takeaways

1. Resample the three primary allocations independently. Explain why the resulting blocks are artificial.
2. Resample participant indices separately by allocation. Which repeated-person covariance disappears?
3. Increase the number of bootstrap draws. Why does this not increase the eight model blocks?
4. Change the prospective assumed standard deviation. How do resolution and materiality probabilities respond?
5. Replace the residual contrast with raw GFC only. Which interpretation becomes unavailable?

**Takeaway:** exposure, sequence support, origin support, participant count, and trained-model replication answer different questions, and only the last one sets the primary denominator. A valid sensitivity carries complete blocks and complete participant profiles. The residual contrast preserves the same pairing. Prospective planning never quietly turns the model-level $n=8$ into something larger.

## Continue learning

[Previous notebook: 14](14_paired_inference.ipynb) | [Lecture](../lectures/15_exposure_and_replication.md) | [Curriculum](../README.md) | [Next notebook: 16](16_reproducible_scientific_evaluators.ipynb)
